# Designing an HR Database

## Step 1: Database Design Requirements

### Data Architect Business Requirements

#### Business Request

#### Business Justification

#### Current Data Management Solution

(The current method of data storage/management)

#### Current Data Available

#### Additional (Future) Data Requests

#### Who Will Own/Manage Database

#### Who Will Have Access to Database

#### Estimated Size of Database and Estimated Annual Growth

#### Sensitive or Restricted Data

### Technical Design Details

#### Data to Be Stored

#### Database Objects

(List of all tables that will be created for the database)

#### Scalability (Replicated Databases/Shard)

#### Flexibility

#### Storage

(In alignment with IT best practices guide)

<h4>Retention</h4>

#### Backup

(In alignment with IT best practices guide)

## Step 2: Relational Database Design

### Conceptual Model

```mermaid

```

### Logical Model

```mermaid

```

### Physical Model

```mermaid

```

## Step 3: Data Contract

### JSON Contract

In [ ]:
CONTRACT_FILE = "data_contract.json"
DATA_FILE     = "HR_Dataset_V2.xlsx"

print(f"Contract : {CONTRACT_FILE}")
print(f"Data     : {DATA_FILE}")

### Data Load/Validation

In [ ]:
import json
import re
import openpyxl
import pandas as pd

try:
    with open(CONTRACT_FILE) as f:
        contract = json.load(f)
        print(f"Contract name    : {contract['name']}")
        print(f"Contract version : {contract['version']}")
        print(f"Fields defined   : {list(contract['columns'].keys())}")
except:
    print(f"Failed to load. Confirm creation of file {CONTRACT_FILE} with required fields.")

In [ ]:
# YOUR CODE HERE

### Connect to AWS

In [ ]:
import boto3

# locate keys in Cloud Resources
AWS_ACCESS_KEY_ID = "<your key here>"
AWS_SECRET_ACCESS_KEY = "<your key here>"
AWS_SESSION_TOKEN = "<your token here>"
AWS_REGION = "us-east-1"

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION,
)

sts = session.client("sts")
identity = sts.get_caller_identity()

print("\n***** AWS Identity *****")
print(identity)

cf = session.client("cloudformation")
stacks = cf.describe_stacks()
stack = stacks["Stacks"][0]
outputs = { o["OutputKey"]: o["OutputValue"] for o in stack["Outputs"] }
db_host = outputs["DBEndpoint"]
db_port = int(outputs["DBPort"])
db_name = outputs["DBName"]
secret_arn = outputs["MasterSecretArn"]

print("\n***** CloudFormation Outputs *****")
print(db_host, db_port, db_name, secret_arn)

rds = session.client("rds", region_name="us-east-1")
response = rds.describe_db_instances(
    DBInstanceIdentifier="udacity-daf-pgres-training-db"
)
db = response["DBInstances"][0]

print("\n***** RDS Database *****")
print("Status:", db["DBInstanceStatus"])
print("Publicly accessible:", db["PubliclyAccessible"])
print("Endpoint:", db["Endpoint"]["Address"])
print("VPC security groups:", db["VpcSecurityGroups"])

sm = session.client("secretsmanager", region_name="us-east-1")
secret_response = sm.get_secret_value(SecretId=secret_arn)
db_secret = json.loads(secret_response["SecretString"])
db_user = db_secret["username"]
db_password = db_secret["password"]

print("\n***** Database Credentials *****")
print(db_user)

In [ ]:
import psycopg2

try:
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode="require",  # important for RDS
    )

    print("Successfully connected to the database")

    with conn.cursor() as cur:
        cur.execute("SELECT version();")
        result = cur.fetchone()

    print("PostgreSQL version:")
    print(result[0])

except Exception as e:
    print("Connection failed")
    print(type(e).__name__, str(e))

### Push Excel Data to a Landing Table

In [ ]:
# YOUR CODE HERE

## Step 4: Create a Physical Database

### DDL

In [ ]:
# YOUR CODE HERE

### DML to Populate Tables

In [ ]:
# YOUR CODE HERE

### CRUD Tests

#### Question 1: Return a list of employees with Job Titles and Department Names

In [ ]:
# YOUR CODE HERE

#### Question 2: Insert Web Programmer as a new job title

In [ ]:
# YOUR CODE HERE

#### Question 3: Correct the job title from web programmer to web developer

In [ ]:
# YOUR CODE HERE

#### Question 4: Delete the job title Web Developer from the database

In [ ]:
# YOUR CODE HERE

#### Question 5: How many employees in each department

In [ ]:
# YOUR CODE HERE

#### Question 6: Write a query that returns current and past jobs (include employee name, job title, department, manager name, start and end date for position) for employee Toni Lembeck.

In [ ]:
# YOUR CODE HERE

#### Question 7: Describe how you would apply table security to restrict access to employee salaries using an SQL server.

In [ ]:
# YOUR CODE HERE

In [ ]:
conn.close()